In [ ]:
import pandas as pd

# Direct link to the classic Telco Churn dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# Load the data into a table (DataFrame)
customer_data = pd.read_csv(url)

# Show the first 5 rows
customer_data.head()

In [ ]:
# Check the data types and look for missing values
customer_data.info()


In [ ]:
# 1. Drop the customerID column 
customer_data = customer_data.drop('customerID', axis=1)

# 2. Force TotalCharges to be numeric (errors='coerce' turns blanks into missing values)
customer_data['TotalCharges'] = pd.to_numeric(customer_data['TotalCharges'], errors='coerce')

# 3. Fill those missing values with 0
customer_data['TotalCharges'] = customer_data['TotalCharges'].fillna(0)

# 4. Show a success message
print("Data cleaned! TotalCharges is now a number.")

In [ ]:
# Translate all remaining text columns into 1s and 0s
customer_data = pd.get_dummies(customer_data, drop_first=True)

# Let's look at the first 5 rows of our fully numbered dataset!
customer_data.head()

In [ ]:
%pip install scikit-learn


In [ ]:
# Import the tools we just installed
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Step 1: Separate the "Features" (Inputs) from the "Target" (What we want to predict)
# We drop the Churn column to get our inputs (X)
X = customer_data.drop('Churn_Yes', axis=1) 
# We isolate the Churn column to get our answers (y)
y = customer_data['Churn_Yes']

# Step 2: Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Step 3: Create the "Brain" (The Random Forest algorithm)
model = RandomForestClassifier(random_state=42)

# Step 4: Train the model! (This is where the math happens)
model.fit(X_train, y_train)

print("Training complete! The model has learned the patterns.")

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Hand the model the final exam (just the customer info)
predictions = model.predict(X_test)

# 2. Grade the exam (Compare its predictions to the real answers)
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.1f}%\n")

# 3. See exactly *how* it made mistakes
print("The Confusion Matrix (Actual vs Predicted):")
print(confusion_matrix(y_test, predictions))

In [ ]:
import pandas as pd

# 1. Ask the model for its importance scores
importance_scores = model.feature_importances_

# 2. Match those scores to the names of our columns
feature_ranking = pd.DataFrame({
    'Warning Sign (Feature)': X.columns,
    'Importance Score': importance_scores
})

# 3. Sort the list from highest to lowest and look at the top 10
feature_ranking = feature_ranking.sort_values(by='Importance Score', ascending=False)
feature_ranking.head(10)

In [ ]:
# Ask the model for the exact probabilities instead of just 1s and 0s
probabilities = model.predict_proba(X_test)

# The model gives us two numbers: [Probability of Staying, Probability of Leaving]
# We just want the second number (Probability of Leaving), which is at index 1
risk_percentages = probabilities[:, 1]

# Let's attach these percentages to our test data so we can see who the high-risk people are
results_table = pd.DataFrame({
    'Customer_Index': X_test.index,
    'Actual_Churn': y_test,
    'Risk_Percentage': risk_percentages * 100 # Multiply by 100 to make it a clean percentage
})

# Show the top 5 highest-risk customers!
results_table.sort_values(by='Risk_Percentage', ascending=False).head(5)

In [ ]:
# 1. Make a copy of our test data so we don't mess up the original
dashboard_data = X_test.copy()

# 2. Add our new Risk Percentages and the Actual answers to this table
dashboard_data['Risk_Percentage'] = risk_percentages * 100
dashboard_data['Actual_Churn'] = y_test

# 3. Sort the whole table to put the highest risk customers at the very top
highest_risk_first = dashboard_data.sort_values(by='Risk_Percentage', ascending=False)

# 4. Let's look at the top 5, but only show the most important warning signs to keep it clean
columns_to_show = ['Risk_Percentage', 'tenure', 'MonthlyCharges'] 
highest_risk_first[columns_to_show].head(5)

In [ ]:
# Save our final results table to a file in your project folder
highest_risk_first.to_csv('churn_risk_data.csv', index=False)
print("Bridge created! Data saved.")